# Machine Learning Foundation: MLOps, Training Pipelines & NN Projects

## Using REAL DATASETS from torchvision

**CIFAR-10** (Module 1): 32×32 color images, 10 classes  
**Fashion MNIST** (Module 3): 28×28 grayscale images, 10 classes

In [ ]:
!pip install mlflow hydra-core torch torchvision scikit-learn pyyaml -q
print('✓ Packages installed')

In [ ]:
import os, yaml, json, numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from pathlib import Path
from typing import Dict, Any
import mlflow
import matplotlib.pyplot as plt

print(f'PyTorch: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')

# PART 1: DEEP LEARNING TRAINING PIPELINES

## Step 1: Create Configuration YAML

In [ ]:
# Create configuration directory
os.makedirs('./configs', exist_ok=True)

config_yaml = """
model:
  num_classes: 10
  input_channels: 3
  conv_filters: [32, 64, 128]
  dropout_rate: 0.3

training:
  batch_size: 128
  num_epochs: 30
  learning_rate: 0.001
  weight_decay: 0.0001
  early_stopping_patience: 5

experiment:
  experiment_name: "cifar10_baseline"
  seed: 42
"""

with open('./configs/config.yaml', 'w') as f:
    f.write(config_yaml.strip())

print('✓ Configuration created')

## Step 2: Load Real CIFAR-10 Dataset

In [ ]:
# Create transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2470, 0.2435, 0.2616]
    )
])

print('Loading CIFAR-10...')
cifar10_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
cifar10_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

print(f'✓ CIFAR-10 loaded: {len(cifar10_train)} train, {len(cifar10_test)} test')
print(f'  Classes: {cifar10_train.classes}')

In [ ]:
# Visualize samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    img, label = cifar10_train[i]
    img = img.clone()
    img[0] = img[0] * 0.2470 + 0.4914
    img[1] = img[1] * 0.2435 + 0.4822
    img[2] = img[2] * 0.2616 + 0.4465
    axes.ravel()[i].imshow(img.permute(1, 2, 0).clamp(0, 1))
    axes.ravel()[i].set_title(cifar10_train.classes[label])
    axes.ravel()[i].axis('off')
plt.tight_layout()
plt.show()

## Step 3: Define CNN and Training Functions

In [ ]:
class CNN(nn.Module):
    def __init__(self, config):
        super().__init__()
        layers = []
        in_ch = 3
        for out_ch in config['model']['conv_filters']:
            layers.extend([
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(),
                nn.MaxPool2d(2, 2),
                nn.Dropout(config['model']['dropout_rate'])
            ])
            in_ch = out_ch
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_ch * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

def train_epoch(model, loader, criterion, opt, device):
    model.train()
    loss_total, correct, total = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        opt.step()
        loss_total += loss.item()
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return loss_total / len(loader), 100 * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    loss_total, correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_total += criterion(out, y).item()
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    return loss_total / len(loader), 100 * correct / total

print('✓ Model and training functions defined')

## Step 4: Load Config and Setup Training

In [ ]:
with open('./configs/config.yaml') as f:
    config = yaml.safe_load(f)

np.random.seed(config['experiment']['seed'])
torch.manual_seed(config['experiment']['seed'])

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Create dataloaders
train_size = int(0.8 * len(cifar10_train))
train_data, val_data = torch.utils.data.random_split(cifar10_train, [train_size, len(cifar10_train) - train_size])

train_loader = DataLoader(train_data, batch_size=128, shuffle=True, num_workers=0)
val_loader = DataLoader(val_data, batch_size=128, num_workers=0)
test_loader = DataLoader(cifar10_test, batch_size=128, num_workers=0)

print(f'✓ Data loaders created (device: {device})')

## Step 5: Train CNN on Real CIFAR-10

In [ ]:
model = CNN(config).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)

print('Training CNN on CIFAR-10...\n')
best_val_acc = 0
history = {'epoch': [], 'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(20):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    history['epoch'].append(epoch)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:3d} | Train: {train_acc:6.2f}% | Val: {val_acc:6.2f}%')

    if val_acc > best_val_acc:
        best_val_acc = val_acc

print(f'\nBest validation accuracy: {best_val_acc:.2f}%')

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['epoch'], history['train_loss'], label='Train')
axes[0].plot(history['epoch'], history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history['epoch'], history['train_acc'], label='Train')
axes[1].plot(history['epoch'], history['val_acc'], label='Val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

# PART 2: COMPUTE RESOURCE MANAGEMENT

In [ ]:
# Cost comparison
gpu_prices = {
    'Colab Pro': 10 / (24 * 30),
    'AWS T4': 0.35,
    'AWS V100': 3.06,
    'GCP T4': 0.29
}

print('\nCost for 5h training × 10 experiments:\n')
for gpu, rate in sorted(gpu_prices.items(), key=lambda x: x[1]):
    cost = 5 * rate * 10
    print(f'{gpu:15s} | ${rate:>6.2f}/hr | Total: ${cost:>7.2f}')

---

# PART 3: NEURAL NETWORK PROJECTS

## Fashion MNIST Classification with Real Data

In [ ]:
# Load Fashion MNIST (real dataset)
fmnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

print('Loading Fashion MNIST...')
fmnist_train = datasets.FashionMNIST(root='./data', train=True, download=True, transform=fmnist_transform)
fmnist_test = datasets.FashionMNIST(root='./data', train=False, download=True, transform=fmnist_transform)

classes = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']
print(f'✓ Fashion MNIST: {len(fmnist_train)} train, {len(fmnist_test)} test')

In [ ]:
# Visualize Fashion MNIST
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    img, label = fmnist_train[i]
    axes.ravel()[i].imshow(img.squeeze(), cmap='gray')
    axes.ravel()[i].set_title(classes[label])
    axes.ravel()[i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Fashion MNIST model
class FashionNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.model(x)

# Setup
fmnist_train_size = int(0.8 * len(fmnist_train))
fmnist_train_data, fmnist_val_data = torch.utils.data.random_split(
    fmnist_train, [fmnist_train_size, len(fmnist_train) - fmnist_train_size]
)

fmnist_train_loader = DataLoader(fmnist_train_data, batch_size=128, shuffle=True, num_workers=0)
fmnist_val_loader = DataLoader(fmnist_val_data, batch_size=128, num_workers=0)
fmnist_test_loader = DataLoader(fmnist_test, batch_size=128, num_workers=0)

fmnist_model = FashionNN().to(device)
print('✓ Fashion MNIST model created')

In [ ]:
# Train Fashion MNIST
print('Training Fashion MNIST classifier...\n')

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(fmnist_model.parameters(), lr=0.001)

fmnist_best_acc = 0
patience = 0

for epoch in range(30):
    train_loss, train_acc = train_epoch(fmnist_model, fmnist_train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(fmnist_model, fmnist_val_loader, criterion, device)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d} | Train: {train_acc:6.2f}% | Val: {val_acc:6.2f}%')

    if val_acc > fmnist_best_acc:
        fmnist_best_acc = val_acc
        patience = 0
    else:
        patience += 1
        if patience >= 5:
            print(f'Early stopping at epoch {epoch + 1}')
            break

print(f'\nBest validation accuracy: {fmnist_best_acc:.2f}%')

In [ ]:
# Test set evaluation
fmnist_model.eval()
test_correct, test_total = 0, 0
with torch.no_grad():
    for x, y in fmnist_test_loader:
        x, y = x.to(device), y.to(device)
        out = fmnist_model(x)
        test_correct += (out.argmax(1) == y).sum().item()
        test_total += y.size(0)

test_accuracy = 100 * test_correct / test_total
print(f'\nTest Set Accuracy: {test_accuracy:.2f}%')

## Compare with Traditional ML (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

print('Training Random Forest baseline...\n')

# Prepare data
X_train = np.vstack([x.view(-1).numpy() for x, _ in fmnist_train_loader])
y_train = np.concatenate([y.numpy() for _, y in fmnist_train_loader])
X_test = np.vstack([x.view(-1).numpy() for x, _ in fmnist_test_loader])
y_test = np.concatenate([y.numpy() for _, y in fmnist_test_loader])

# Train
rf = RandomForestClassifier(n_estimators=50, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_acc = accuracy_score(y_test, rf.predict(X_test)) * 100

print(f'\n=== COMPARISON ===')
print(f'Random Forest:  {rf_acc:.2f}%')
print(f'Neural Network: {test_accuracy:.2f}%')
print(f'Advantage:      {test_accuracy - rf_acc:+.2f}%')

---

# SUMMARY

## Key Learnings ✓

✓ **Real Datasets**: CIFAR-10, Fashion MNIST from torchvision  
✓ **Configuration Management**: YAML for hyperparameters  
✓ **Reproducibility**: Fixed seeds, version control  
✓ **Training Pipelines**: Configurable, loggable workflows  
✓ **Cost Awareness**: GPU pricing and optimization  
✓ **Project Development**: End-to-end with MLOps  
✓ **Baseline Comparison**: Traditional ML vs Neural Networks  

---

**This notebook is fully compatible with Google Colab!**